In [0]:
# =============================================================
# 05_BATCH_INFERENCE — Setup Autónomo (sin %run)
# =============================================================
import json
import boto3
import os
import sys

# --- Credenciales ---
try:
    config = {
        "aws_access_key": dbutils.secrets.get(scope="aws", key="access_key"),
        "aws_secret_key": dbutils.secrets.get(scope="aws", key="secret_key"),
    }
    print("✅ Credenciales desde Databricks Secrets.")
except Exception:
    try:
        with open("aws_secrets.json", "r") as f:
            config = json.load(f)
        print("✅ Credenciales desde aws_secrets.json.")
    except FileNotFoundError:
        raise SystemExit("❌ Credenciales no disponibles.")

BUCKET = "bronce-scrap-date"
S3_OPTIONS = {
    "fs.s3a.access.key": config["aws_access_key"],
    "fs.s3a.secret.key": config["aws_secret_key"],
    "fs.s3a.endpoint": "s3.amazonaws.com",
}

print(f"✅ Bucket: {BUCKET}")


In [0]:
import pandas as pd
import numpy as np
import pickle

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from src.ml.scorer import score_dataframe

GOLD_PATH = f"s3a://{BUCKET}/gold/app_inmuebles/"
SCORED_PATH = f"s3a://{BUCKET}/gold/app_inmuebles_scored/"

print(f"📖 Cargando Parquet desde: {GOLD_PATH}")
df_gold_spark = spark.read.format("parquet").options(**S3_OPTIONS).load(GOLD_PATH)

print("⬇️ Bajando a Pandas...")
df_pandas = df_gold_spark.toPandas()
print(f"✅ {len(df_pandas)} registros cargados.")


In [0]:
%pip install xgboost

In [0]:
import io

print(f"🔍 Buscando modelo en s3://{BUCKET}/models/ ...")

s3_client = boto3.client(
    's3',
    aws_access_key_id=config["aws_access_key"],
    aws_secret_access_key=config["aws_secret_key"],
    region_name='us-east-1'
)

response = s3_client.list_objects_v2(Bucket=BUCKET, Prefix='models/')
if 'Contents' not in response:
    raise FileNotFoundError(f"No hay archivos en s3://{BUCKET}/models/")

model_files = [
    obj for obj in response['Contents']
    if obj['Key'].endswith('.pkl') and 'bundle_v' in obj['Key']
]
if not model_files:
    raise FileNotFoundError("No se encontró bundle .pkl en S3.")

latest = max(model_files, key=lambda x: x['LastModified'])
print(f"📦 Descargando: {latest['Key']}")

body = s3_client.get_object(Bucket=BUCKET, Key=latest['Key'])['Body'].read()
bundle = pickle.load(io.BytesIO(body))

print(f"✅ Modelo {bundle.get('model_version', 'v?')} cargado.")


In [0]:
print("⚙️ Generando rentabilidad...")
df_scored_pandas = score_dataframe(df_pandas, bundle)
print("✅ Inferencia completada.")
display(df_scored_pandas[["titulo", "city_token", "precio_num", "precio_predicho", "rentabilidad_potencial", "estado_inversion"]].head(10))


In [0]:
print("⬆️ Guardando resultado...")

for col in df_scored_pandas.columns:
    if df_scored_pandas[col].dtype == "object":
        df_scored_pandas[col] = df_scored_pandas[col].fillna("").astype(str)

df_scored_spark = spark.createDataFrame(df_scored_pandas)

print(f"💾 Guardando en: {SCORED_PATH}")
writer = df_scored_spark.write.format("delta").mode("overwrite").options(**S3_OPTIONS).option("overwriteSchema", "true")
writer.save(SCORED_PATH)

print("🎉 Misión Inferencia Batch Finalizada.")
